# Branch Operations: Tool Usage & Function Calling

This notebook explores lionagi's powerful function calling capabilities, showing how AI agents can interact with external tools and perform complex operations beyond text generation.

**Key Concepts:**
- **Tool Integration**: Connect AI agents to external functions and APIs
- **Automatic Function Selection**: Let the AI choose which tools to use based on context
- **Parallel Function Execution**: Handle multiple function calls simultaneously
- **Structured Action Flow**: Track function calls, parameters, and results

**Function Calling in Practice:**

The `operate()` method enables AI agents to analyze problems and automatically select appropriate tools to solve them. This bridges the gap between natural language understanding and concrete actions.

**How Function Calling Works:**
1. **Problem Analysis**: The AI examines the given context and instruction
2. **Tool Selection**: Based on available tools, the AI decides which functions to call
3. **Parameter Extraction**: The AI determines the correct parameters for each function
4. **Execution**: Functions are called with the AI-generated parameters
5. **Result Integration**: Function outputs are incorporated back into the conversation

This creates a powerful paradigm where AI agents can reason about problems and take concrete actions to solve them.

In [1]:
question = "There are [basketball, football, backpack, water bottle, strawberry, tennis ball, rockets]. each comes in four different colors, what is the number of the only unique kinds of ball(not other stuff)?"

question2 = "There are three fruits in total, each with 2 different colors, how many unique kinds of fruits are there?"

context1 = {"Question1": question, "question2": question2}

In [2]:
# define a function


def multiply(number1: float, number2: float):
    """
    Perform multiplication on two numbers.

    Args:
        number1: First number to multiply.
        number2: Second number to multiply.

    Returns:
        The product of number1 and number2.

    """
    return number1 * number2

In [3]:
from lionagi import Branch, iModel

branch = Branch(
    system="you are asked to perform as a function picker and parameter provider",
    tools=multiply,
    chat_model=iModel(model="openai/gpt-5.4-mini"),
)

task = "Think step by step, understand the following basic math question and provide parameters for function calling. with parallel processing"

In [4]:
# request actions by flagging actions=True
from pydantic import BaseModel

class Answer(BaseModel):
    q1: int
    q2: int

out = await branch.operate(
    instruction=task,
    context=context1,
    actions=True,
    lndl=True,
    response_format=Answer
)

print(out)
print(branch.messages[-1].rendered)
# print(out.action_requests)
# print(out.action_responses)

q1=8 q2=6 action_required=None action_requests=[ActionRequestModel(function='multiply', arguments={'number1': 2, 'number2': 4}), ActionRequestModel(function='multiply', arguments={'number1': 3, 'number2': 2})] action_responses=[ActionResponseModel(function='multiply', arguments={'number1': 2, 'number2': 4}, output=8), ActionResponseModel(function='multiply', arguments={'number1': 3, 'number2': 2}, output=6)]
Function: multiply
Arguments:
  number1: 3
  number2: 2
Output: 6


**Understanding the Output:**

The output shows two key components:

**Action Requests**: The AI's plan for solving the problems
- Question 1: Identified 3 types of balls × 4 colors = 12 total
- Question 2: Identified 3 fruits × 2 colors = 6 total

Notice how the AI automatically determined which calculations were needed and called the multiply function in parallel to solve both questions efficiently.

In [5]:
branch.to_df()

,created_at,role,content,id,sender,recipient,metadata
0,1.778210e+09,system,{'system_message': 'you are asked to perform a...,a7a906bf-0a37-41dd-93fc-c0fb4636bd19,system,bb330e58-db70-428c-9980-c5929b18e72d,{'lion_class': 'lionagi.protocols.messages.sys...
1,1.778210e+09,user,"{'images': [], 'prompt_context': [{'Question1'...",ad86740a-2edc-4e04-81e1-2fc7d053a8e4,user,bb330e58-db70-428c-9980-c5929b18e72d,{'lion_class': 'lionagi.protocols.messages.ins...
2,1.778210e+09,assistant,{'assistant_response': '<lact q1 a>multiply(nu...,19feaf52-3b25-4a09-85ea-58945f907d9f,bb330e58-db70-428c-9980-c5929b18e72d,user,{'model_response': {'id': 'chatcmpl-Dd5uDYoS0g...
3,1.778210e+09,action,{'action_response_id': '92be628e-28f4-4a66-925...,6ff4ae5e-e928-4919-a526-c4974b5e9291,bb330e58-db70-428c-9980-c5929b18e72d,e33dc529-438b-4e80-bfc0-2126d0704c7b,{'lion_class': 'lionagi.protocols.messages.act...
4,1.778210e+09,action,"{'function': 'multiply', 'action_request_id': ...",92be628e-28f4-4a66-9252-60fab3abb34c,e33dc529-438b-4e80-bfc0-2126d0704c7b,bb330e58-db70-428c-9980-c5929b18e72d,{'lion_class': 'lionagi.protocols.messages.act...
5,1.778210e+09,action,{'action_response_id': '034809ce-858c-4ac7-a69...,bac342f5-30f9-4607-afa7-e6e3c2d39824,bb330e58-db70-428c-9980-c5929b18e72d,e33dc529-438b-4e80-bfc0-2126d0704c7b,{'lion_class': 'lionagi.protocols.messages.act...
6,1.778210e+09,action,"{'function': 'multiply', 'action_request_id': ...",034809ce-858c-4ac7-a697-00e712086a8e,e33dc529-438b-4e80-bfc0-2126d0704c7b,bb330e58-db70-428c-9980-c5929b18e72d,{'lion_class': 'lionagi.protocols.messages.act...


In [6]:
print(branch.messages[2].rendered)

<lact q1 a>multiply(number1=2, number2=4)</lact>
<lact q2 b>multiply(number1=3, number2=2)</lact>

OUT{q1: [a], q2: [b]}


In [7]:
out

AnswerResponse(q1=8, q2=6, action_required=None, action_requests=[ActionRequestModel(function='multiply', arguments={'number1': 2, 'number2': 4}), ActionRequestModel(function='multiply', arguments={'number1': 3, 'number2': 2})], action_responses=[ActionResponseModel(function='multiply', arguments={'number1': 2, 'number2': 4}, output=8), ActionResponseModel(function='multiply', arguments={'number1': 3, 'number2': 2}, output=6)])